In [19]:
import seaborn as sns
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler


In [20]:
ins_data = pd.read_csv("./data/insurance.csv")


In [21]:
ins_data.isnull().mean()

age         0.0
sex         0.0
bmi         0.0
children    0.0
smoker      0.0
region      0.0
charges     0.0
dtype: float64

Since there are no null values , we don't need to do any imputations to the dataset.

#### Selecting label and features:

In [22]:
label = "charges" #the thing to be predicted
y = ins_data[label]
features  = [i for i in ins_data.columns if i!=label]
X = ins_data[features]
X_train,X_test,y_train,y_test = train_test_split(X,y,random_state=1)

#### Selecting categorical and numerical columns:

In [23]:
num_cols = [i for i in X.columns if X[i].dtype in ["float64","int64"]]
cate_cols = [i for i in X.columns if X[i].dtype =='object' and X[i].nunique() < 15]


### Setting up pipelines:

In [24]:
##column transforming pipeline setup (One hot encodes the categorical data)
cate_coltrans = Pipeline(steps=[
    ('OneHotEncoder',OneHotEncoder(handle_unknown='ignore',sparse_output=False,drop='first'))
       
    ]
) 
##preprocessor setup


preprocessor = ColumnTransformer(
    transformers=[
        ('cate',cate_coltrans,cate_cols),
        
        ('num', StandardScaler(), num_cols)
    ]
) 

## FIrst model Test:
model = DecisionTreeRegressor(random_state=1)
my_model = Pipeline(
    steps=[
        ('preprocessor',preprocessor),
        ('model',model)
    ]
)
my_model.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cate',
                                                  Pipeline(steps=[('OneHotEncoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['sex', 'smoker', 'region']),
                                                 ('num', StandardScaler(),
                                                  ['age', 'bmi',
                                                   'children'])])),
                ('model', DecisionTreeRegressor(random_state=1))])

In [25]:
pred = my_model.predict(X_test) ##predictions made

In [26]:
mae = mean_absolute_error(y_test,pred)
print("R2:", r2_score(y_test, pred))
print("MAE:", mae)

R2: 0.6546028352049624
MAE: 3335.2235362686565


### Hyperparameter tuning the Decision tree regressor :

In [27]:
def mae_dtree(max_leaf_nodes,X_train,X_test,y_train,y_test):
    new_model = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes,random_state=1)
    my_model = Pipeline(
        steps = [
            ('preprocessor',preprocessor),
            ('model',new_model)
        ]
    )
    my_model.fit(X_train,y_train)
    pred=my_model.predict(X_test)
    mae = mean_absolute_error(y_test,pred)
    return mae
    

In [28]:
max_leaf_nodes = [i*10 for i in range(50) if i!=0]
dtree_mae = dict()
for i in max_leaf_nodes:
    dtree_mae[i] = mae_dtree(i,X_train,X_test,y_train,y_test)
    print(f"For {i} nodes the Mean absolute error is : {dtree_mae[i]}")
mini = min(dtree_mae,key=dtree_mae.get)
print(f"Best mae is for {mini} nodes which is {dtree_mae[mini]}")

For 10 nodes the Mean absolute error is : 2610.9649508189077
For 20 nodes the Mean absolute error is : 2681.52732443655
For 30 nodes the Mean absolute error is : 2518.9728450097655
For 40 nodes the Mean absolute error is : 2537.17223738635
For 50 nodes the Mean absolute error is : 2679.214091574972
For 60 nodes the Mean absolute error is : 2728.4368047786857
For 70 nodes the Mean absolute error is : 2734.5858281350825
For 80 nodes the Mean absolute error is : 2825.4941757895676
For 90 nodes the Mean absolute error is : 2874.91856653197
For 100 nodes the Mean absolute error is : 3032.72689817883
For 110 nodes the Mean absolute error is : 3039.2315265892967
For 120 nodes the Mean absolute error is : 3134.146212452978
For 130 nodes the Mean absolute error is : 3148.9149959371935
For 140 nodes the Mean absolute error is : 3233.968888077906
For 150 nodes the Mean absolute error is : 3311.4190795684026
For 160 nodes the Mean absolute error is : 3341.935024269184
For 170 nodes the Mean absolu

### Testing out RandomForestRegressor:

In [29]:
new_model = RandomForestRegressor(
    random_state=1
)

new_pipe_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', new_model)
    ]
)

new_pipe_model.fit(X_train, y_train)

pred = new_pipe_model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("R2:", r2_score(y_test, pred))


MAE: 2740.287350620179
R2: 0.8247016713457875


In [30]:
def mae_RandomTree(max_leaf_nodes,X_train,X_test,y_train,y_test):
    new_model = RandomForestRegressor(max_leaf_nodes=max_leaf_nodes,random_state=1)
    my_model = Pipeline(
        steps = [
            ('preprocessor',preprocessor),
            ('model',new_model)
        ]
    )
    my_model.fit(X_train,y_train)
    pred=my_model.predict(X_test)
    mae = mean_absolute_error(y_test,pred)
    return mae

In [31]:
max_rand_leaf_nodes = [i*10 for i in range(50) if i!=0]
Random_tree_reg_mae = dict()
for i in max_rand_leaf_nodes:   
    Random_tree_reg_mae[i]=mae_RandomTree(i,X_train,X_test,y_train,y_test)
    print(f"For {i} nodes the mae is {Random_tree_reg_mae[i]}")
print(f"Best mae is for {min(Random_tree_reg_mae,key=Random_tree_reg_mae.get)}")

For 10 nodes the mae is 2559.083176994543
For 20 nodes the mae is 2512.3035346452007
For 30 nodes the mae is 2503.6106602745367
For 40 nodes the mae is 2510.1412508893054
For 50 nodes the mae is 2525.77523695198
For 60 nodes the mae is 2546.982825127743
For 70 nodes the mae is 2596.278225872799
For 80 nodes the mae is 2638.611842920179
For 90 nodes the mae is 2675.471955268787
For 100 nodes the mae is 2710.100900074988
For 110 nodes the mae is 2741.954327721712
For 120 nodes the mae is 2762.5887690205336
For 130 nodes the mae is 2776.7476907819196
For 140 nodes the mae is 2778.494125001619
For 150 nodes the mae is 2782.0564000870268
For 160 nodes the mae is 2779.8341911681737
For 170 nodes the mae is 2776.686270913677
For 180 nodes the mae is 2772.705387687366
For 190 nodes the mae is 2768.6117317971334
For 200 nodes the mae is 2765.891341752774
For 210 nodes the mae is 2763.0311839104575
For 220 nodes the mae is 2760.40896975177
For 230 nodes the mae is 2759.750000853068
For 240 nodes

### Two regression models were trained and evaluated to predict insurance charges.
##### Models tested:
- Decision Tree Regressor
- Random Forest Regressor

##### Final Model: Random Forest Regressor (Without Cross Validation):
> Mean Absolute Error: 2503.6106602745367

##### The two models came very close in terms of MAE. 


### Applying cross Validation on all the models now:

In [34]:
model_lis = {'Decision Tree reg':my_model,'Random Forest Reg':new_pipe_model} ##my_model = Decision Tree reg | new_pipe_model = Random Forest Reg | new_lin_pipe_model = Linear Reg
model_cv_rec=dict()
for name,model in model_lis.items():
    scores = -1*cross_val_score(model, X, y, cv=5, scoring="neg_mean_absolute_error")
    print(f"{name}")
    print("Average MAE:", scores.mean())
    print("Std:", scores.std())
    print("-----")
    

Decision Tree reg
Average MAE: 3083.5150823231816
Std: 275.2998667133933
-----
Random Forest Reg
Average MAE: 2712.909668981002
Std: 145.20753324362568
-----


### Final Model: Random Forest Regressor

- Best Hyperparameter:
max_leaf_nodes = 30

#### Performance:

- MAE: 2503
- R2: 0.73

#### Random Forest outperformed Decision Tree and provided more stable predictions.
